#### 3.3.2 Матрица связей

**Входы**

- train и типы переменных

**Выходы**

- Коэффициенты связи для всех пар переменных

**Что делаем**

Для разных типов данных выберем подходящую меру связи и соберём результаты в матрицу, включая пары с loan_status

##### Как измеряем связи

Одна мера не подходит сразу для чисел, неупорядоченных категорий и грейдов, поэтому выбираем метод по типу пары

| Тип переменной | Числовой | Порядковый | Номинальный |
| --- | --- | --- | --- |
| **Числовой** | Пирсон $r$ | Спирмен $\rho$ | Нормализованная взаимная информация |
| **Порядковый** | Спирмен $\rho$ | Спирмен $\rho$ | Краскел — Уоллис $\varepsilon^2$ |
| **Номинальный** | Нормализованная взаимная информация | Краскел — Уоллис $\varepsilon^2$ | V Крамера |

Бинарные столбцы считаем номинальными. У грейда сохраняем порядок A < B < C < D < E < F < G

Для каждой пары используем строки без пропусков в её двух столбцах. Соберём коэффициенты в одну матрицу

###### Общая матрица

Зададим типы признаков и подготовим матрицу для коэффициентов связи

In [ ]:
from itertools import combinations_with_replacement
from scipy.stats import chi2_contingency, kruskal
from sklearn.feature_selection import mutual_info_classif


# Общая матрица: инициализируем один раз перед последовательными этапами
association_columns = train.columns
association_matrix = pd.DataFrame(
    np.nan, index=association_columns, columns=association_columns, dtype=float,
)

# Для выбора мер связи используем смысловые типы из описаний полей.
association_types = pd.Series({
    'person_age': 'числовой',
    'person_income': 'числовой',
    'person_home_ownership': 'номинальный',
    'person_emp_length': 'числовой',
    'loan_intent': 'номинальный',
    'loan_grade': 'порядковый',
    'loan_amnt': 'числовой',
    'loan_int_rate': 'числовой',
    'loan_status': 'бинарный',
    'loan_percent_income': 'числовой',
    'cb_person_default_on_file': 'бинарный',
    'cb_person_cred_hist_length': 'числовой',
}).reindex(association_columns)
numerical_columns = association_types.index[association_types.eq('числовой')].tolist()


###### Числовые пары — Pearson

Коэффициент Pearson измеряет силу и направление линейной связи, принимает значения от −1 до 1

In [ ]:
# Числовые пары
pearson_matrix = train[numerical_columns].corr(method='pearson', min_periods=2)
association_matrix.loc[numerical_columns, numerical_columns] = pearson_matrix


###### Номинальные пары — Cramér’s V

Cramér’s V рассчитываем по таблице сопряжённости и статистике хи-квадрат без коррекции смещения. Диапазон — от 0 до 1, бинарные признаки включаем в номинальные

In [ ]:
def cramers_v(left, right):
    paired = pd.concat([left, right], axis=1).dropna()
    if paired.empty:
        return np.nan
    table = pd.crosstab(paired.iloc[:, 0], paired.iloc[:, 1])
    # Удаляем ненаблюдаемые категории, если столбцы имеют dtype category
    table = table.loc[table.sum(axis=1).gt(0), table.sum(axis=0).gt(0)]
    k, m = table.shape
    if min(k, m) < 2:
        return np.nan
    chi2 = chi2_contingency(table, correction=False)[0]
    return np.sqrt(chi2 / (table.to_numpy().sum() * min(k - 1, m - 1)))


nominal_columns = association_types.index[
    association_types.isin(['номинальный', 'бинарный'])
].tolist()

# Номинальные пары, включая бинарные
for left, right in combinations_with_replacement(nominal_columns, 2):
    value = cramers_v(train[left], train[right])
    association_matrix.loc[left, right] = value
    association_matrix.loc[right, left] = value


###### Числовые и порядковые пары — Spearman

Spearman оценивает монотонную связь по рангам с усреднением совпадений. Диапазон — от −1 до 1. Для loan_grade задаём порядок A < B < C < D < E < F < G; для порядковых пар используем ту же меру

In [ ]:
# Явный порядок категорий, без изменения исходного train
ordinal_orders = {'loan_grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G']}
ordinal_columns = association_types.index[association_types.eq('порядковый')].tolist()

for ordinal in ordinal_columns:
    order = ordinal_orders[ordinal]
    observed = train[ordinal].dropna()
    if not observed.isin(order).all():
        raise ValueError(f'Неизвестные категории в {ordinal}')
    ordinal_values = train[ordinal].map({grade: rank for rank, grade in enumerate(order)})
    for numerical in numerical_columns:
        paired = pd.DataFrame({
            'numerical': train[numerical],
            'ordinal': ordinal_values,
        }).dropna()
        value = paired.corr(method='spearman', min_periods=2).loc['numerical', 'ordinal']
        association_matrix.loc[numerical, ordinal] = value
        association_matrix.loc[ordinal, numerical] = value

# Пары порядковых переменных, включая диагональ
ordinal_data = pd.DataFrame({
    column: train[column].map({value: rank for rank, value in enumerate(ordinal_orders[column])})
    for column in ordinal_columns
}, index=train.index)
association_matrix.loc[ordinal_columns, ordinal_columns] = ordinal_data.corr(
    method='spearman', min_periods=2,
)


###### Числовые и номинальные пары — NMI

Нормализуем оценку взаимной информации по энтропии номинального признака: $I(X;Y)/H(Y)$. Мера учитывает различия средних, разбросов и форм распределений. При нулевой энтропии нормализация не определена

In [ ]:
def numerical_nominal_nmi(numerical, nominal):
    paired = pd.DataFrame({'x': numerical, 'y': nominal}).dropna()
    if paired.empty:
        return np.nan, np.nan, np.nan
    labels, _ = pd.factorize(paired['y'], sort=False)
    counts = np.bincount(labels)
    probabilities = counts / counts.sum()
    entropy = float(-(probabilities * np.log(probabilities)).sum())
    if entropy <= 0:
        return np.nan, entropy, np.nan
    if paired['x'].nunique() <= 1:
        return 0.0, entropy, 0.0
    if counts.min() < 2:
        return np.nan, entropy, np.nan
    mi = mutual_info_classif(
        paired[['x']], labels, discrete_features=False,
        n_neighbors=3, random_state=42,
    )[0]
    ratio = mi / entropy
    return float(mi), entropy, float(np.clip(ratio, 0.0, 1.0))


###### Заполнение числовых × номинальных пар

Оцениваем взаимную информацию без разбиения числового признака на категории. Нормализованные оценки ограничиваем интервалом от 0 до 1

In [ ]:
# Числовые × номинальные
nmi_records = []
for numerical in numerical_columns:
    for nominal in nominal_columns:
        mi, entropy, value = numerical_nominal_nmi(train[numerical], train[nominal])
        association_matrix.loc[numerical, nominal] = value
        association_matrix.loc[nominal, numerical] = value
        nmi_records.append({
            'Numerical': numerical, 'Nominal': nominal,
            'MI estimate': mi, 'H(Y)': entropy,
            'MI / H(Y)': mi / entropy if entropy > 0 else np.nan,
            'NMI': value,
        })

nmi_estimates = pd.DataFrame(nmi_records).set_index(['Numerical', 'Nominal'])


###### Порядковые и номинальные пары — Краскел — Уоллис

Критерий сравнивает ранги между группами, учитывая совпадения. Он использует порядок категорий без предположения о равных расстояниях между ними. В матрицу записываем размер эффекта

$$\varepsilon^2=\frac{H-k+1}{n-k},$$

где $k$ — число групп, $n$ — число наблюдений. Метод может пропустить зависимость, если распределения различаются, а средние ранги групп совпадают

In [ ]:
def ordinal_nominal_kruskal(ordinal, nominal, order):
    paired = pd.DataFrame({'ordinal': ordinal, 'nominal': nominal}).dropna()
    paired['rank_code'] = paired['ordinal'].map(
        {category: rank for rank, category in enumerate(order)}
    )
    groups = [
        group['rank_code'].to_numpy(dtype=float)
        for _, group in paired.groupby('nominal', observed=True, sort=False)
    ]
    k, n = len(groups), len(paired)
    if k < 2 or n <= k or paired['rank_code'].nunique() < 2:
        return np.nan, k, n, np.nan
    h = float(kruskal(*groups).statistic)
    effect = (h - k + 1) / (n - k)
    return h, k, n, float(effect)


###### Заполнение порядковых × номинальных пар

Сохраняем исходную оценку ε². Только при выборе цвета ограничиваем её интервалом от 0 до 1, поскольку выборочная оценка может быть отрицательной

In [ ]:
# Порядковые × номинальные
kruskal_records = []
for ordinal in ordinal_columns:
    for nominal in nominal_columns:
        h, k, n, effect = ordinal_nominal_kruskal(
            train[ordinal], train[nominal], ordinal_orders[ordinal],
        )
        association_matrix.loc[ordinal, nominal] = effect
        association_matrix.loc[nominal, ordinal] = effect
        kruskal_records.append({
            'Ordinal': ordinal, 'Nominal': nominal,
            'H': h, 'k': k, 'n': n, 'epsilon_squared': effect,
        })

kruskal_effects = pd.DataFrame(kruskal_records).set_index(['Ordinal', 'Nominal'])


###### Как читать матрицу

Каждая ячейка показывает связь двух переменных. Смотрим только нижнюю половину: верхняя повторяла бы те же пары\
Чем насыщеннее цвет, тем сильнее связь по выбранной мере. У Pearson и Spearman знак в подписи показывает направление: плюс — значения в среднем растут вместе, минус — меняются в противоположных направлениях. Для остальных мер направления нет\
Палитры различают методы; числа разных методов напрямую не сравниваем. Пустая ячейка означает, что коэффициент не удалось определить

In [ ]:
# include: _code/66_plot_associations.py